# DS-01 – Score de risque de refus des colis payables à la livraison
TuniShop – dossier TS-DATA-DSC-001 – v1.0 – 14/10/2026

**Question** : peut-on estimer, au moment de l'expédition, le risque qu'un colis payable à la livraison soit refusé (BM-11, PR-2) ?

**Protocole** : découpage temporel (apprentissage 2023-2024, test 2025) ; variables connues au plus tard à l'expédition ; comparaison avec la règle métier « première commande ou panier > 400 DT » (référence PR-2).

In [1]:
import os, json, warnings
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
# Dossier donnees_reference du jeu TuniShop (variable d'environnement ou chemin relatif)
R = os.environ.get("TUNISHOP_REF_DIR", "../donnees_reference") + "/"
FIG = os.environ.get("TUNISHOP_FIG_DIR", "figures") + "/"; os.makedirs(FIG, exist_ok=True)
SEED = 42
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

## 1. Construction du jeu d'étude
Une ligne par colis payable à la livraison, livré ou revenu en échec. L'historique du client ne compte que les commandes **antérieures** (pas de fuite d'information).

In [2]:
c=pd.read_csv(R+'commandes.csv',parse_dates=['date_commande']); l=pd.read_csv(R+'livraisons.csv',parse_dates=['date_expedition','date_livraison_prevue'])
lc=pd.read_csv(R+'lignes_commande.csv'); p=pd.read_csv(R+'produits.csv'); k=pd.read_csv(R+'categories.csv'); g=pd.read_csv(R+'ref_gouvernorats.csv'); cal=pd.read_csv(R+'dim_calendrier.csv',parse_dates=['date'])
cl=pd.read_csv(R+'clients.csv',parse_dates=['date_inscription'])
d=l.merge(c,on='commande_id',suffixes=('','_c'))
d=d[(d.mode_paiement=='paiement_a_la_livraison')&d.statut_livraison.isin(['livree','echec_retour_expediteur'])].copy()
d['y']=(d.statut_livraison=='echec_retour_expediteur').astype(int)
# historique client strictement antérieur
h=l.merge(c[['commande_id','client_id','date_commande','mode_paiement']],on='commande_id').sort_values('date_commande')
h['refus']=(h.statut_livraison=='echec_retour_expediteur')&(h.mode_paiement=='paiement_a_la_livraison')
h['n_prec']=h.groupby('client_id').cumcount(); h['refus_prec']=h.groupby('client_id').refus.cumsum()-h.refus
d=d.merge(h[['commande_id','n_prec','refus_prec']],on='commande_id')
# panier: part électronique, nb lignes
lp=lc.merge(p[['produit_id','categorie_id']]).merge(k[['categorie_id','categorie']])
mix=lp.assign(elec=lp.categorie.isin(['Électronique & High-Tech','Électroménager'])*lp.montant_ligne_ttc).groupby('commande_id').agg(elec=('elec','sum'),tot=('montant_ligne_ttc','sum'))
d=d.merge((mix.elec/mix.tot).rename('part_elec').reset_index(),on='commande_id',how='left')
d=d.merge(g[['gouvernorat_id','region','zone_livraison']],left_on='gouvernorat_livraison_id',right_on='gouvernorat_id',how='left')
d['jour']=d.date_commande.dt.normalize(); d=d.merge(cal[['date','periode_commerciale']],left_on='jour',right_on='date',how='left')
d['heure']=d.date_commande.dt.hour; d['jsem']=d.date_commande.dt.dayofweek
d['delai_promis']=(d.date_livraison_prevue-d.date_expedition.dt.normalize()).dt.days
d['log_montant']=np.log1p(d.montant_total_ttc); d['coupon']=d.code_coupon.notna().astype(int)
d=d.merge(cl[['client_id','type_client','date_inscription']],on='client_id'); d['anciennete_j']=(d.date_commande-d.date_inscription).dt.days.clip(lower=0)
num=['log_montant','nb_articles','part_elec','n_prec','refus_prec','heure','jsem','delai_promis','anciennete_j','coupon','est_premiere_commande']
cat=['region','zone_livraison','canal','appareil','transporteur_id','entrepot_id','periode_commerciale','type_client']
d['est_premiere_commande']=d.est_premiere_commande.astype(int)
for x in cat: d[x]=d[x].fillna('NA').astype(str)


In [3]:
tr=d[d.date_commande<'2025-01-01']; te=d[d.date_commande>='2025-01-01']
pd.DataFrame({'lignes':[len(tr),len(te)],'taux_de_refus':[tr.y.mean(),te.y.mean()]},index=['apprentissage 2023-2024','test 2025']).round(4)

,lignes,taux_de_refus
apprentissage 2023-2024,17552,0.0797
test 2025,13190,0.0804


## 2. Modèles
Référence : règle métier (score ordinal : première commande compte double, panier > 400 DT). Candidats : régression logistique régularisée, gradient boosting d'arbres.

In [4]:
res={}
base=(te.est_premiere_commande*2+(te.montant_total_ttc>400)).values  # règle de référence (score ordinal)
res['Règle : 1re commande, panier > 400 DT']=base
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
lr=make_pipeline(pre,LogisticRegression(max_iter=2000,C=0.5)); lr.fit(tr[num+cat],tr.y); res['Régression logistique']=lr.predict_proba(te[num+cat])[:,1]
pre2=ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)])
gb=make_pipeline(pre2,HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=50,random_state=42)); gb.fit(tr[num+cat],tr.y); res['Gradient boosting']=gb.predict_proba(te[num+cat])[:,1]


## 3. Évaluation sur 2025
Métriques : aire sous la courbe ROC (AUC), précision moyenne (AP), part des refus captés dans les 20 % de colis les plus risqués. Intervalle de confiance de l'AUC par bootstrap (500 tirages).

In [5]:
rng=np.random.default_rng(SEED); y=te.y.values; lignes=[]
for n,s in res.items():
    o=np.argsort(-s); top=int(len(s)*0.2); rec20=y[o[:top]].sum()/y.sum()
    b=[]
    for _ in range(500):
        i=rng.integers(0,len(y),len(y)); b.append(roc_auc_score(y[i],np.asarray(s)[i]))
    lignes.append(dict(modele=n,AUC=roc_auc_score(y,s),IC95_bas=np.percentile(b,2.5),IC95_haut=np.percentile(b,97.5),AP=average_precision_score(y,s),rappel_top20=rec20))
resultats=pd.DataFrame(lignes).round(3); resultats

,modele,AUC,IC95_bas,IC95_haut,AP,rappel_top20
0,"Règle : 1re commande, panier > 400 DT",0.570,0.555,0.585,0.099,0.307
1,Régression logistique,0.561,0.541,0.578,0.107,0.294
2,Gradient boosting,0.567,0.548,0.585,0.104,0.314


In [6]:
fig,ax=plt.subplots(figsize=(6,5))
for n,s in res.items():
    f,t,_=roc_curve(y,s); ax.plot(f,t,label=f"{n} (AUC {roc_auc_score(y,s):.3f})")
ax.plot([0,1],[0,1],'k--',lw=0.8); ax.set_xlabel('Taux de faux positifs'); ax.set_ylabel('Taux de vrais positifs'); ax.set_title('Refus de colis – courbes ROC (test 2025)'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG+'ds01_roc.png',dpi=150)

## 4. Facteurs de risque observés

In [7]:
fact=pd.DataFrame({'Première commande':[te[te.est_premiere_commande==1].y.mean(),te[te.est_premiere_commande==0].y.mean()],
 'Panier > 400 DT':[te[te.montant_total_ttc>400].y.mean(),te[te.montant_total_ttc<=400].y.mean()],
 'Client avec refus antérieur':[te[te.refus_prec>0].y.mean(),te[te.refus_prec==0].y.mean()]},index=['oui','non']).T.round(3); fact

,oui,non
Première commande,0.130,0.070
Panier > 400 DT,0.101,0.078
Client avec refus antérieur,0.067,0.083


## 5. Conclusion
Voir la cellule suivante : le critère de succès de PR-2 (AUC du modèle supérieure à celle de la règle, intervalle de confiance compris) est évalué automatiquement.

In [8]:
ref=resultats.set_index('modele').loc['Règle : 1re commande, panier > 400 DT','AUC']; meilleur=resultats.sort_values('AUC').iloc[-1]
succes=bool(meilleur.IC95_bas>ref)
json.dump(dict(resultats=resultats.to_dict('records'),facteurs=fact.to_dict(),succes=succes,n_train=len(tr),n_test=len(te),taux_test=float(te.y.mean())),open(FIG+'ds01_resultats.json','w'),ensure_ascii=False,default=float)
print('Meilleur modèle :',meilleur.modele,'AUC',meilleur.AUC,'| règle',ref,'| critère atteint :',succes)

Meilleur modèle : Règle : 1re commande, panier > 400 DT AUC 0.57 | règle 0.57 | critère atteint : False
